# Criteo Uplift v2.1 — can we work with all 14 million rows, and where does the time actually go?

**The question.** Before any modeling, two things have to be true: the data has to be what we think it is, and processing all of it has to be affordable. This notebook establishes both, on the complete dataset — every row, no sampling — and measures where the time is actually spent.

**Why that second part matters more than it sounds.** A full-data analysis like this one issues many queries against the same file. If every query re-decompresses and re-parses the whole compressed CSV from the beginning, that reading cost can dominate everything else — including diagnostics that look, on paper, like the expensive part. Section 4 measures this directly, by timing one real analytical workload against the file as shipped and against a converted columnar copy, before any other full-data diagnostic runs. Every section after it uses whichever storage format that measurement supports.

**What you will find here**

1. Why ranking by conversion probability and ranking by treatment effect are different problems
2. What machine this ran on, and what the input actually is
3. What is in the file: schema, completeness, group sizes, outcome counts
4. Whether the raw compressed file is the bottleneck — and what to do about it
5. Whether the treated and untreated groups are comparable
6. Whether their full distributions differ, not just their averages
7. How many rows are indistinguishable, and whether numeric precision invents collisions
8. Where the time went, and what the modeling work that follows must respect

**What is not here.** No model is trained and nothing is predicted. Every number is computed at run time — nothing is hardcoded from a previous session.

## 1. Why response modeling and uplift modeling are different questions

Response modeling estimates **P(conversion | features)** — how likely is this person to convert?

Uplift modeling estimates **E[conversion if treated] − E[conversion if not treated]** among people with the same features — how much does treating this person *change* the outcome?

These rank people differently, and the difference is not subtle. Consider two customers:

- **Customer A** converts with probability 0.90 if shown the ad, and 0.89 if not. She was going to buy anyway. Her uplift is about +1 percentage point.
- **Customer B** converts with probability 0.20 if shown the ad, and 0.10 if not. His uplift is +10 percentage points.

A response model ranks A far above B. An uplift model ranks B far above A. Spend the budget on A and most of it buys conversions that would have happened regardless.

The only reason this question can be asked at all is that treatment here was **randomly assigned**. Without randomization, treated and untreated people would differ systematically, and the gap between their conversion rates would reflect who they are rather than what the ad did.

In [ ]:
# Illustrative only — these four customers are invented and are NOT from the Criteo data.
import pandas as pd
from IPython.display import Markdown, display

illustration = pd.DataFrame({
    'customer': ['A', 'B', 'C', 'D'],
    'p_convert_if_treated': [0.90, 0.20, 0.55, 0.05],
    'p_convert_if_not_treated': [0.89, 0.10, 0.50, 0.01],
})
illustration['uplift'] = illustration['p_convert_if_treated'] - illustration['p_convert_if_not_treated']
illustration['rank_by_response'] = illustration['p_convert_if_treated'].rank(ascending=False).astype(int)
illustration['rank_by_uplift'] = illustration['uplift'].rank(ascending=False).astype(int)
display(illustration)

display(Markdown('''
**What this shows.** The two rankings disagree almost completely. Customer A is the best target by
conversion probability and the *worst* target by uplift; customer B is the reverse.

**What this does not show.** Nothing yet about the Criteo data — these are invented numbers used only
to make the distinction concrete.

**Why it matters next.** Everything below asks whether this dataset can support the second kind of
ranking, and whether it can do so at full scale.
'''))

## 2. What are we running on, and what did we get?

**Question.** What machine is this, what versions are installed, and which input file was found?

**Why it matters.** Every timing later in this notebook is meaningless without knowing the machine that produced it. A conclusion like *this is affordable* is a statement about a specific amount of CPU and RAM, and it does not transfer to a different one unless the numbers travel with it.

Note that the query engine is left to configure **itself** rather than being capped at some fixed thread count or memory limit. Imposing an artificial cap would answer the wrong question — it would tell us whether the work is feasible under that cap, not whether it is feasible on this platform. Whatever the engine chose is reported below.

In [ ]:
# Setup: locate the input, open the query engine, and report the environment.
# Runs on Kaggle with the Criteo uplift dataset attached, and locally, with no repository dependency.
import platform
import sys
import tempfile
import time
from contextlib import contextmanager
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

try:
    import psutil
except ImportError:
    psutil = None

FEATURES = [f'f{i}' for i in range(12)]
TREATMENT, OUTCOME, SECONDARY, DIAGNOSTIC = 'treatment', 'conversion', 'visit', 'exposure'
TIMINGS = {}
PROC = psutil.Process() if psutil else None


def sql_literal(value):
    '''Quote a path for use inside a SQL string literal.'''
    quote = chr(39)
    return quote + str(value).replace(quote, quote * 2) + quote


@contextmanager
def timed(label):
    '''Record wall time and the change in memory held by this process.'''
    rss_before = PROC.memory_info().rss if PROC else 0
    start = time.perf_counter()
    yield
    seconds = time.perf_counter() - start
    rss_after = PROC.memory_info().rss if PROC else 0
    TIMINGS[label] = {'seconds': seconds, 'rss_delta_bytes': rss_after - rss_before}


def find_input():
    '''Kaggle input first, then common local locations.'''
    for root in [Path('/kaggle/input'), Path('data/raw'), Path('../data/raw'), Path('.')]:
        if not root.is_dir():
            continue
        for pattern in ('**/*.csv.gz', '**/*.csv'):
            for candidate in sorted(root.glob(pattern)):
                if 'criteo' in candidate.name.lower():
                    return candidate
    raise FileNotFoundError(
        'Could not find the Criteo uplift file. On Kaggle, attach the Criteo uplift dataset; '
        'locally, place it under data/raw/.'
    )


INPUT_PATH = find_input()
# Scratch space for a derived file. On Kaggle this is /kaggle/working; locally it is the OS temp
# directory, deliberately NOT anywhere inside the repository.
WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path(tempfile.gettempdir())
PARQUET_PATH = WORK_DIR / 'criteo-uplift-v2.1-feasibility.parquet'

con = duckdb.connect(database=':memory:')  # no thread or memory cap: let the engine choose
con.execute(f'CREATE VIEW raw AS SELECT * FROM read_csv_auto({sql_literal(INPUT_PATH)})')

chosen_threads = con.execute("SELECT current_setting('threads')").fetchone()[0]
chosen_memory = con.execute("SELECT current_setting('memory_limit')").fetchone()[0]

environment = [
    ('Python', sys.version.split()[0]),
    ('Platform', platform.platform()),
    ('Processor', platform.processor() or 'not reported'),
    ('Logical CPUs', psutil.cpu_count(logical=True) if psutil else 'psutil unavailable'),
    ('Physical CPUs', psutil.cpu_count(logical=False) if psutil else 'psutil unavailable'),
    ('Total RAM (GiB)', f'{psutil.virtual_memory().total / 2 ** 30:.1f}' if psutil else 'psutil unavailable'),
    ('Available RAM (GiB)', f'{psutil.virtual_memory().available / 2 ** 30:.1f}' if psutil else 'psutil unavailable'),
    ('duckdb', duckdb.__version__),
    ('pandas', pd.__version__),
    ('numpy', np.__version__),
    ('Engine threads (auto)', chosen_threads),
    ('Engine memory limit (auto)', chosen_memory),
    ('Input file', str(INPUT_PATH)),
    ('Input format', 'gzip-compressed CSV' if INPUT_PATH.suffix == '.gz' else 'plain CSV'),
    ('Input size (MiB)', f'{INPUT_PATH.stat().st_size / 2 ** 20:,.1f}'),
    ('Scratch directory', str(WORK_DIR)),
]
display(pd.DataFrame(environment, columns=['property', 'value']))

display(Markdown(f'''
**What this shows.** The work below runs on {chosen_threads} engine threads with a memory limit of
`{chosen_memory}`, both chosen by the engine rather than imposed. The input is a
{'gzip-compressed CSV' if INPUT_PATH.suffix == '.gz' else 'plain CSV'} of
{INPUT_PATH.stat().st_size / 2 ** 20:,.1f} MiB.

**What this does not show.** Nothing about the data itself yet, and nothing about whether these
resources are sufficient — that is measured, not assumed, in section 4.

**Why it matters next.** Every duration reported later belongs to this machine and this configuration.
Read them together.
'''))

## 3. What is actually in the file?

**Question.** Are the expected columns present and correctly typed, is anything missing, how large is each treatment group, and how rare is the outcome?

**Why it matters.** These are the facts every later section depends on. They are gathered here in a **single pass** over the file, because at this point the compressed CSV is still the only thing we have and each pass over it is expensive.

**What a row is, and what it is not.** Each row is one released observation from the experiment. It is **not** a person: the public schema has no user identifier, so two identical rows might be two different people or the same person twice, and the data cannot tell us. The twelve features `f0`–`f11` are **anonymous**; Criteo does not publish what they represent, so any story about what `f3` means would be invented.

The roles are fixed for the whole analysis:

| Field | Role |
|---|---|
| `f0`–`f11` | the only model inputs |
| `treatment` | randomly assigned; 1 = ad assigned, 0 = not |
| `conversion` | the primary outcome |
| `visit` | a secondary outcome, analysed separately; never an input |
| `exposure` | whether the ad was actually seen — diagnostic only, never an input |

**Why `exposure` is never an input and never a filter.** Whether someone actually saw the ad happens *after* assignment and depends on their own behaviour. Using it as a feature, or keeping only exposed rows, would swap a randomized comparison for a self-selected one, and the difference it produced would reflect who those people are at least as much as what the ad did. The effect of interest throughout is the effect of *being assigned* the ad.

In [ ]:
# One pass over the compressed input: schema, size, completeness, group sizes, outcome counts.
schema = con.execute('DESCRIBE raw').df()[['column_name', 'column_type']]
present = [c for c in FEATURES + [TREATMENT, OUTCOME, SECONDARY, DIAGNOSTIC]
           if c in schema['column_name'].tolist()]

missing_exprs = ', '.join(f'COUNT(*) - COUNT({c}) AS miss_{c}' for c in present)
with timed('3. Sanity pass over compressed input'):
    sanity = con.execute(f'''
        SELECT COUNT(*) AS n_rows,
               COUNT_IF({TREATMENT} = 1) AS treated_n,
               COUNT_IF({TREATMENT} = 0) AS control_n,
               COUNT_IF({OUTCOME} = 1) AS conversions_n,
               COUNT_IF({SECONDARY} = 1) AS visits_n,
               COUNT_IF({TREATMENT} = 1 AND {OUTCOME} = 1) AS treated_conv,
               COUNT_IF({TREATMENT} = 0 AND {OUTCOME} = 1) AS control_conv,
               {missing_exprs}
        FROM raw
    ''').df().iloc[0]

total_n = int(sanity['n_rows'])
treated_n, control_n = int(sanity['treated_n']), int(sanity['control_n'])
treated_conv, control_conv = int(sanity['treated_conv']), int(sanity['control_conv'])
treated_share = treated_n / total_n
treated_rate, control_rate = treated_conv / treated_n, control_conv / control_n
overall_rate = int(sanity['conversions_n']) / total_n
missing_total = int(sum(int(sanity[f'miss_{c}']) for c in present))

roles = {c: 'model input' for c in FEATURES}
roles.update({TREATMENT: 'randomized assignment', OUTCOME: 'primary outcome',
              SECONDARY: 'secondary outcome (never an input)',
              DIAGNOSTIC: 'diagnostic only (never an input)'})
display(pd.DataFrame({
    'field': present,
    'stored type': [schema.loc[schema['column_name'] == c, 'column_type'].iloc[0] for c in present],
    'role': [roles[c] for c in present],
    'missing values': [int(sanity[f'miss_{c}']) for c in present],
}))

cells = pd.DataFrame({
    'treatment': [0, 0, 1, 1],
    'conversion': [0, 1, 0, 1],
    'rows': [control_n - control_conv, control_conv, treated_n - treated_conv, treated_conv],
})
cells['within_group_share'] = cells['rows'] / cells['treatment'].map({0: control_n, 1: treated_n})
display(cells)

display(Markdown(f'''
**What this shows.** {total_n:,} rows and {len(schema)} columns, with **{missing_total:,}** missing
values in total across every field used here. The assignment is heavily lopsided:
{treated_share:.1%} of rows were assigned the ad and {1 - treated_share:.1%} were not
({treated_n:,} against {control_n:,}). Conversion is rare — {overall_rate:.4%} overall,
{control_rate:.4%} for the unassigned group and {treated_rate:.4%} for the assigned group.
All four combinations of group and outcome are populated: {bool((cells["rows"] > 0).all())}.

**What this does not show.** A complete, well-typed table says nothing about whether assignment was
genuinely random, nor whether the features were measured before treatment. The gap between the two
conversion rates is a comparison of two averages, not a per-person effect, and it says nothing about
*which* people any effect is concentrated in.

**Why it matters next.** Rarity, not row count, is the binding constraint: it is the
{int(sanity["conversions_n"]):,} conversions that limit how finely the data can be sliced. And this
single pass took {TIMINGS['3. Sanity pass over compressed input']['seconds']:.1f}s — which raises the
question the next section answers.
'''))

## 4. Is the file format the real bottleneck?

**Question.** How long does a realistic analytical query take against the compressed CSV, and would converting once to a columnar format pay for itself?

**Why it matters.** This is the question that governs every section after it. A compressed CSV must be decompressed and re-parsed from the beginning on *every* query, and it is row-oriented, so asking for twelve columns costs the same as asking for all sixteen. A columnar format is parsed once, stores each column separately, and keeps summary statistics per block. If the difference is large, then an analysis that felt too expensive to run at full scale was never really about the statistics.

**How this is measured.** The same workload is timed three ways: on the compressed input as it ships, then again on a converted copy. The workload is a real one — the per-group mean and variance of all twelve features, which is exactly what section 5 needs — not an artificial microbenchmark. The conversion is timed too, so its cost is counted honestly rather than hidden.

The converted file is **scratch**: a throwaway artifact for this notebook's own use. It is not a curated or published dataset and nothing downstream should treat it as one.

In [ ]:
# Time the real workload on the compressed input, then convert once (reusing an existing copy).
balance_agg = ', '.join(f'AVG({c}) AS {c}_mean, VAR_SAMP({c}) AS {c}_var' for c in FEATURES)
workload = f'SELECT {TREATMENT}, {balance_agg} FROM {{table}} GROUP BY {TREATMENT}'

with timed('4. Balance workload on compressed CSV'):
    con.execute(workload.format(table='raw')).fetchall()
compressed_seconds = TIMINGS['4. Balance workload on compressed CSV']['seconds']

reused = PARQUET_PATH.exists()
if reused:
    existing_rows = con.execute(
        f'SELECT COUNT(*) FROM read_parquet({sql_literal(PARQUET_PATH)})').fetchone()[0]
    reused = existing_rows == total_n
    if not reused:
        PARQUET_PATH.unlink()

if reused:
    conversion_seconds = 0.0
    conversion_note = 'A converted copy from an earlier run was reused, so no conversion cost was paid.'
else:
    with timed('4. One-time conversion to columnar format'):
        con.execute(
            f'COPY (SELECT * FROM raw) TO {sql_literal(PARQUET_PATH)} '
            '(FORMAT PARQUET, COMPRESSION ZSTD)'
        )
    conversion_seconds = TIMINGS['4. One-time conversion to columnar format']['seconds']
    conversion_note = f'The conversion ran once and took {conversion_seconds:.1f}s.'

con.execute(f'CREATE OR REPLACE VIEW data AS SELECT * FROM read_parquet({sql_literal(PARQUET_PATH)})')
print(f'Columnar copy: {PARQUET_PATH}')
print(f'Size: {PARQUET_PATH.stat().st_size / 2 ** 20:,.1f} MiB '
      f'(input was {INPUT_PATH.stat().st_size / 2 ** 20:,.1f} MiB)')
print(conversion_note)

In [ ]:
# Same workload against the columnar copy, plus an uncompressed CSV only if one already exists.
with timed('4. Balance workload on columnar copy'):
    con.execute(workload.format(table='data')).fetchall()
columnar_seconds = TIMINGS['4. Balance workload on columnar copy']['seconds']

comparison = [('compressed CSV (as shipped)', INPUT_PATH.stat().st_size / 2 ** 20, compressed_seconds)]

plain_csv = INPUT_PATH.with_suffix('') if INPUT_PATH.suffix == '.gz' else None
if plain_csv is not None and plain_csv.is_file():
    con.execute(f'CREATE OR REPLACE VIEW plain AS SELECT * FROM read_csv_auto({sql_literal(plain_csv)})')
    with timed('4. Balance workload on uncompressed CSV'):
        con.execute(workload.format(table='plain')).fetchall()
    comparison.append(('uncompressed CSV (found locally)', plain_csv.stat().st_size / 2 ** 20,
                       TIMINGS['4. Balance workload on uncompressed CSV']['seconds']))
    plain_note = ''
else:
    plain_note = (' An uncompressed CSV was not present, so that row is absent — the dataset ships '
                  'compressed and one was not manufactured just to fill in the table.')

comparison.append(('columnar copy', PARQUET_PATH.stat().st_size / 2 ** 20, columnar_seconds))
formats = pd.DataFrame(comparison, columns=['format', 'size (MiB)', 'workload seconds'])
formats['times slower than columnar'] = formats['workload seconds'] / columnar_seconds
display(formats)

saved_per_pass = compressed_seconds - columnar_seconds
break_even = (conversion_seconds / saved_per_pass) if saved_per_pass > 0 else float('inf')

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.barh(formats['format'], formats['workload seconds'], color='#4C78A8')
ax.set_xlabel('Seconds for one identical analytical query over all 13.98M rows')
ax.set_title('Cost of the same question, asked of three storage formats')
for i, value in enumerate(formats['workload seconds']):
    ax.text(value, i, f' {value:.1f}s', va='center')
ax.set_xlim(0, float(formats['workload seconds'].max()) * 1.15)
fig.tight_layout()
plt.show()

display(Markdown(f'''
**What this shows.** The identical query takes {compressed_seconds:.1f}s against the compressed CSV and
{columnar_seconds:.1f}s against the columnar copy — **{compressed_seconds / columnar_seconds:.0f} times
faster**. Each later pass therefore saves {saved_per_pass:.1f}s. Set against a one-time conversion cost
of {conversion_seconds:.1f}s, the conversion repays itself after **{break_even:.2f}** analytical passes.
{plain_note}

**What this does not show.** These durations belong to this machine and this engine configuration, and
the conversion cost is paid again on any fresh session that starts with empty scratch space. Nor is the
columnar copy a curated dataset — it is a throwaway built by this notebook for its own use.

**Why it matters next.** The storage format is the actual bottleneck being measured here, not the
statistics: reading and re-parsing a compressed file on every query dominates the cost of whatever that
query computes. Every section from here on queries the columnar copy instead, and the full-data
diagnostics that follow — balance, distribution shape, repeated-profile counts — cost seconds rather
than minutes as a direct consequence.
'''))

## 5. Do the assigned and unassigned groups look alike?

**Question.** How similar are the two groups on each of the twelve features?

**Why it matters.** If treatment really was assigned at random, the two groups should look statistically alike on everything measured beforehand. A large systematic gap would be a strong signal that something upstream is broken. Two measures are reported here, because each catches something the other misses:

- the **standardized mean difference**, which expresses the gap between group means in pooled standard deviations, so features on very different scales become comparable;
- the **variance ratio**, which asks whether the groups have similar spread and not merely similar centres.

Both come from a **single pass** that computes the count, mean and variance of all twelve features for both groups at once — not one pass per feature.

**How to read the ±0.10 line.** A dashed line at ±0.10 appears on the chart. It is a rule of thumb from the literature, drawn for orientation only; nothing here passes or fails because of it.

**What balance cannot do.** Groups that look alike are *consistent with* random assignment. They do not prove it — that is a claim about how the experiment was run, which balance can support but never establish. Balance also says nothing about factors that were never measured. With nearly fourteen million rows, conventional significance tests would flag differences far too small to matter, so they are not used.

In [ ]:
# One pass computes count, mean and variance for every feature in both groups.
moment_agg = ', '.join(
    f'COUNT({c}) AS {c}_n, AVG({c}) AS {c}_mean, VAR_SAMP({c}) AS {c}_var' for c in FEATURES)
with timed('5. Balance moments, all features, one pass'):
    moments = con.execute(
        f'SELECT {TREATMENT}, {moment_agg} FROM data GROUP BY {TREATMENT} ORDER BY {TREATMENT}'
    ).df().set_index(TREATMENT)

balance_rows = []
for c in FEATURES:
    mean_treated, mean_control = moments.loc[1, f'{c}_mean'], moments.loc[0, f'{c}_mean']
    var_treated, var_control = moments.loc[1, f'{c}_var'], moments.loc[0, f'{c}_var']

    # Standardized mean difference, written out rather than hidden in a helper.
    pooled_sd = np.sqrt((var_treated + var_control) / 2)
    smd = (mean_treated - mean_control) / pooled_sd if pooled_sd > 0 else np.nan
    variance_ratio = var_treated / var_control if var_control > 0 else np.nan

    balance_rows.append({
        'feature': c,
        'n (assigned)': int(moments.loc[1, f'{c}_n']),
        'n (unassigned)': int(moments.loc[0, f'{c}_n']),
        'mean (assigned)': mean_treated,
        'mean (unassigned)': mean_control,
        'SMD': smd,
        'abs SMD': abs(smd),
        'variance ratio': variance_ratio,
    })

balance = pd.DataFrame(balance_rows)
display(balance)

worst = balance.loc[balance['abs SMD'].idxmax()]
display(Markdown(f'''
**What this shows.** The largest gap between group means is **{worst['abs SMD']:.4f}** pooled standard
deviations, on `{worst['feature']}`. Variance ratios lie between
**{balance['variance ratio'].min():.3f}** and **{balance['variance ratio'].max():.3f}**. All twelve
features were summarized in one pass taking
{TIMINGS['5. Balance moments, all features, one pass']['seconds']:.1f}s.

**What this does not show.** This cannot prove that assignment was random, cannot speak to anything
unmeasured, and cannot rule out differences confined to regions of the feature space too small to move
a global average.

**Why it matters next.** Means and variances can still hide a difference in distribution *shape*. That
is what section 6 looks for.
'''))

In [ ]:
# The same balance evidence, drawn.
ordered = balance.sort_values('SMD')
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))

axes[0].barh(ordered['feature'], ordered['SMD'], color='#4C78A8')
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].axvline(0.10, color='#E45756', linestyle='--', linewidth=1,
                label='plus or minus 0.10 rule-of-thumb reference')
axes[0].axvline(-0.10, color='#E45756', linestyle='--', linewidth=1)
axes[0].set_xlabel('Standardized mean difference (assigned minus unassigned)')
axes[0].set_title('Difference in feature means between groups')
axes[0].legend(loc='lower right', fontsize=8)

ratio_ordered = balance.sort_values('variance ratio')
axes[1].barh(ratio_ordered['feature'], ratio_ordered['variance ratio'], color='#72B7B2')
axes[1].axvline(1.0, color='black', linewidth=0.8)
axes[1].set_xlabel('Variance ratio (assigned / unassigned)')
axes[1].set_title('Difference in feature spread between groups')

fig.tight_layout()
plt.show()

## 6. Do the full distributions differ, not just the averages?

**Question.** For each feature, what is the largest gap between the two groups' cumulative distributions?

**Why it matters.** Two groups can share a mean and a variance and still be shaped differently. This measure — the largest vertical distance between the two cumulative distribution curves — catches that. It is the **secondary** diagnostic here; the standardized mean difference in section 5 remains primary.

**Why it is run on the full data.** Twelve per-feature passes of this kind would be expensive against the compressed CSV, where each feature means another full decompression and re-parse. Against the columnar copy established in section 4, the same twelve calculations cost only a few seconds, so there is no reason to approximate them on a sample. The measured cost is reported below alongside the result, so the choice can be checked rather than taken on trust.

This is one place where a per-feature loop is unavoidable: each feature needs its own ordering of its own distinct values, which cannot be shared across features.

In [ ]:
# Largest gap between the two groups' cumulative distributions, per feature, on the full data.
def distribution_gap(feature):
    return con.execute(f'''
        WITH value_counts AS (
            SELECT {feature} AS value,
                   COUNT_IF({TREATMENT} = 0)::DOUBLE AS n_unassigned,
                   COUNT_IF({TREATMENT} = 1)::DOUBLE AS n_assigned
            FROM data
            WHERE {feature} IS NOT NULL
            GROUP BY {feature}
        ), cumulative AS (
            SELECT value,
                   SUM(n_unassigned) OVER (ORDER BY value ROWS UNBOUNDED PRECEDING)
                       / SUM(n_unassigned) OVER () AS cdf_unassigned,
                   SUM(n_assigned) OVER (ORDER BY value ROWS UNBOUNDED PRECEDING)
                       / SUM(n_assigned) OVER () AS cdf_assigned
            FROM value_counts
        )
        SELECT MAX(ABS(cdf_assigned - cdf_unassigned)) FROM cumulative
    ''').fetchone()[0]


with timed('6. Distribution gaps, all 12 features, full data'):
    gaps = pd.DataFrame(
        [{'feature': c, 'largest distribution gap': distribution_gap(c)} for c in FEATURES])
gap_seconds = TIMINGS['6. Distribution gaps, all 12 features, full data']['seconds']
display(gaps)

worst_gap = gaps.loc[gaps['largest distribution gap'].idxmax()]
fig, ax = plt.subplots(figsize=(8, 4.0))
ordered_gaps = gaps.sort_values('largest distribution gap')
ax.barh(ordered_gaps['feature'], ordered_gaps['largest distribution gap'], color='#72B7B2')
ax.set_xlabel('Largest vertical gap between the two cumulative distribution curves')
ax.set_title('Difference in distribution shape between groups')
fig.tight_layout()
plt.show()

display(Markdown(f'''
**What this shows.** The largest distribution gap is
**{worst_gap['largest distribution gap']:.4f}** on `{worst_gap['feature']}`, computed on all
{total_n:,} rows for all twelve features in **{gap_seconds:.1f}s**.

**What this does not show.** Like the means and variances before it, agreement here is consistent with
random assignment without proving it, and says nothing about anything unmeasured. A gap is a
descriptive quantity, not a test that anything passed.

**Why it matters next.** Because this cost {gap_seconds:.1f}s rather than several minutes, it stays a
full-data measurement instead of an approximation — no sample, no seed, no caveat about which rows were
looked at.
'''))

## 7. How many rows are indistinguishable, and does precision invent collisions?

**Question.** How often do rows share identical values, under three increasingly strict definitions — all twelve features; features plus assigned group; features plus group plus outcome? And separately: does storing the features at lower numeric precision *create* matches that are not in the data?

**Why it matters.** A model literally cannot distinguish two rows whose features are identical, so how common that is affects what any model can learn from them. The precision question matters because if rounding merges genuinely distinct rows, then a decision as mundane as a storage type would quietly change what the data says.

**Why no rows are removed.** It is tempting to call repeated profiles duplicate users and delete them. That would be a mistake twice over. First, the data is anonymous and carries no person identifier, so identical values are not evidence of the same person — with twelve coarse features across nearly fourteen million rows, coincidental matches are expected even if every row is a different individual. Second, deleting rows because their values repeat quietly changes which population the conclusions describe, invisibly. Every row is kept. Rows sharing features but differing in group or outcome are not contradictions; they are exactly the comparisons an uplift model learns from.

**One measurement caveat.** Rows are grouped by a 64-bit hash of their values rather than by comparing every column directly, which is what makes this practical at full scale. A very small number of groups could therefore be coincidental hash collisions rather than genuine matches.

Each grouping below is computed **exactly once**, including the full-precision grouping, which also serves as the baseline for the reduced-precision comparison — recomputing it a second time would double the most expensive operation in this section for no benefit.

In [ ]:
# Each grouping runs once; the precision comparison reuses the full-precision result rather than redoing it.
def profile_groups(columns, source='data'):
    row = con.execute(f'''
        WITH groups AS (
            SELECT hash({', '.join(columns)}) AS profile, COUNT(*) AS n
            FROM {source} GROUP BY profile
        )
        SELECT COUNT(*) FILTER (WHERE n > 1),
               COALESCE(SUM(n) FILTER (WHERE n > 1), 0),
               MAX(n)
        FROM groups
    ''').fetchone()
    return {'repeated groups': int(row[0]),
            'rows in repeated groups': int(row[1]),
            'rows beyond the first': int(row[1]) - int(row[0]),
            'largest group': int(row[2])}


results = {}
with timed('7. Repeated-profile groupings'):
    results['all twelve features'] = profile_groups(FEATURES)
    results['features + assigned group'] = profile_groups(FEATURES + [TREATMENT])
    results['features + group + outcome'] = profile_groups(FEATURES + [TREATMENT, OUTCOME])

reduced = '(SELECT ' + ', '.join(f'CAST({c} AS FLOAT) AS {c}' for c in FEATURES) + ' FROM data)'
with timed('7. Reduced-precision comparison'):
    results['all twelve features, reduced precision'] = profile_groups(FEATURES, source=reduced)

table = pd.DataFrame(results).T.reset_index().rename(columns={'index': 'grouped by'})
table['share of all rows'] = table['rows in repeated groups'] / total_n
display(table)

full_precision = results['all twelve features']
low_precision = results['all twelve features, reduced precision']
invented = low_precision['rows beyond the first'] - full_precision['rows beyond the first']
group_seconds = (TIMINGS['7. Repeated-profile groupings']['seconds']
                 + TIMINGS['7. Reduced-precision comparison']['seconds'])

display(Markdown(f'''
**What this shows.** On all twelve features, **{full_precision['rows in repeated groups']:,} rows**
({full_precision['rows in repeated groups'] / total_n:.2%} of the data) share their profile with at
least one other row, across {full_precision['repeated groups']:,} groups, the largest holding
{full_precision['largest group']:,} rows. Adding the assigned group, and then the outcome, splits these
into progressively smaller groups. Reducing numeric precision raises the count of rows beyond the first
from {full_precision['rows beyond the first']:,} to {low_precision['rows beyond the first']:,} —
**{invented:,} additional rows** ({invented / total_n:.4%}) that are distinguishable at full precision
and not at reduced precision. All four groupings together took {group_seconds:.1f}s.

**What this does not show.** Not one of these numbers counts *people*. Without a person identifier there
is no way to separate one individual appearing repeatedly from many individuals who happen to look
identical on twelve anonymous features. The counts also inherit the hash caveat above.

**Why it matters next.** The {invented:,} rows that only collide at reduced precision are an artifact of
rounding, not a property of the data — which is a concrete reason to keep the features at full precision
downstream. And repeated profiles carrying different outcomes are informative rather than contradictory,
which is why every row is retained.
'''))

## 8. Where did the time actually go?

**Question.** Now that everything has run, which operations were the real bottlenecks?

**Why it matters.** This is the notebook's governing question, and it can only be answered at the end, from measurements rather than intuition. The answer determines what the modeling work that follows can afford to do casually and what needs thought.

In [ ]:
# Consolidated runtime for every timed section.
runtime = pd.DataFrame([
    {'operation': label,
     'seconds': record['seconds'],
     'memory change (MiB)': record['rss_delta_bytes'] / 2 ** 20}
    for label, record in TIMINGS.items()
]).sort_values('seconds', ascending=False).reset_index(drop=True)
runtime['share of measured time'] = runtime['seconds'] / runtime['seconds'].sum()
display(runtime)

total_seconds = float(runtime['seconds'].sum())
on_compressed = float(runtime.loc[runtime['operation'].str.contains('compressed'), 'seconds'].sum())
peak_rss = f'{PROC.memory_info().rss / 2 ** 30:.2f} GiB' if PROC else 'not measured'

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.barh(runtime['operation'][::-1], runtime['seconds'][::-1], color='#4C78A8')
ax.set_xlabel('Seconds')
ax.set_title('Measured cost of each stage')
fig.tight_layout()
plt.show()

display(Markdown(f'''
**What this shows.** All measured work totalled **{total_seconds:.1f}s**, of which {on_compressed:.1f}s
({on_compressed / total_seconds:.0%}) was spent reading the compressed CSV in the two places it could
not be avoided. Memory in use at the end was {peak_rss}. Every full-data statistical diagnostic —
balance, distribution gaps, and all four repeated-profile groupings — completed in seconds once the data
was in a columnar form.

**What this does not show.** These are this machine's numbers under the engine settings reported in
section 2, and the memory figure is the amount held at the end rather than a sampled peak. A different
machine will give different durations, though the *ratio* between formats should survive.

**Why it matters next.** The bottleneck was never the statistics. The modeling work that follows can
treat full-data diagnostics as cheap, provided it reads from a columnar copy and does not re-derive the
same grouping twice.
'''))

## Key findings

- **The bottleneck is the storage format, not the analysis.** The same query is dramatically faster against a columnar copy than against the compressed CSV, and a one-time conversion repays itself within the first analytical pass. What looks like statistically expensive analysis is really the cost of re-parsing a compressed file on every question.
- **Full-scale processing is affordable.** Every diagnostic here runs over all 13.98M rows. Nothing is sampled and nothing was dropped for cost.
- **The experiment is complete and well formed.** Every field used is fully populated, and all four combinations of assigned group and outcome occur.
- **Assignment is heavily lopsided and conversion is rare.** The smaller group caps precision for anything estimated within it, and the count of conversions — not the count of rows — is what limits how finely the data can be sliced.
- **The two groups look alike** on means, variances, and distribution shape. This is consistent with random assignment; it does not prove it, and no threshold was used to decide it.
- **A meaningful share of rows are indistinguishable on the twelve features**, and reducing numeric precision manufactures further collisions that are not in the data — a concrete reason to work at full precision and to keep every row.
- **Individual treatment effects remain unmeasurable.** Only one of the two possible outcomes is ever observed per row, so there is no per-row ground truth. Everything downstream estimates conditional averages and should be read that way.

**What the modeling work that follows must respect.** Read from a columnar copy, not the raw archive. Compute each grouping once. Expect the smaller assigned group and the rare outcome to drive precision. Treat any mention of T-Learner, X-Learner or Causal Forest as motivation for work not yet done — no model has been trained here, and nothing above is a result about one.